# OSI Import, the agent way (CLI + MCP)

**TL;DR:** the [companion notebook](osi_import_nb.ipynb) imported an OSI config with the Python library. This one walks the *same* demo the way an **AI agent wired to SLayer** experiences it — ingest the config with two **command-line** commands, then explore and query the result through SLayer's **MCP tools** (the exact tool calls Claude would make).

Steps (everything lands in a gitignored `.cache/`):

1. **Build the data & reference answers** — the same tiny retail DuckDB; gold SQL computed up front.
2. **Ingest via the CLI** — `slayer datasources create`, then `slayer import-osi`.
3. **Connect the MCP server** and explore — `models_summary`, `inspect`, `search`.
4. **Query via the MCP `query` tool** — verified against gold.

Fully offline — no network needed.

## Configure SLayer as an MCP server in Claude

In real use you register SLayer once with your agent; Claude then spawns it on demand and calls its tools. With [uv](https://docs.astral.sh/uv/) installed:

```bash
claude mcp add slayer -- uvx --from 'motley-slayer[advanced_search]' slayer mcp --storage /path/to/store
```

or, after a permanent install (`uv tool install 'motley-slayer[advanced_search]'`):

```bash
claude mcp add slayer -- slayer mcp --storage /path/to/store
```

Point `--storage` at the folder holding your datasource + models — here, this demo's `.cache/slayer_models`. Agents that take a JSON config use the equivalent:

```json
{
  "mcpServers": {
    "slayer": {
      "command": "uvx",
      "args": ["--from", "motley-slayer[advanced_search]", "slayer", "mcp", "--storage", "/path/to/store"]
    }
  }
}
```

`slayer mcp` speaks JSON-RPC over **stdio**, so there's no HTTP URL to curl. This notebook drives the same tools **in-process** via `create_mcp_server(...).call_tool(name, arguments)` — the identical `tools/call` request an stdio client issues — so you can watch each call run. Full guide: [MCP Setup — AI Agents](../../getting-started/mcp.md).

## Step 1 — Build the data & reference answers

The same retail DuckDB as the companion notebook. We compute the gold answers **first**, before the importer or the MCP engine opens the file (DuckDB won't share a read-write file across connections). The gold SQL lives in `setup_osi.compute_gold`, shared by both notebooks.

In [1]:
import json
import os
import warnings
import shutil
import subprocess
import sys

import pandas as pd

# FastMCP emits a benign PydanticJsonSchemaWarning when it builds the
# tool JSON schemas; silence it so the tool listing stays clean.
warnings.filterwarnings("ignore", message=".*not JSON serializable.*")

# The setup helper lives next to this notebook.
sys.path.insert(0, os.getcwd())

from setup_osi import (
    build_shop_duckdb,
    compute_gold,
    OSI_CONFIG,
    DB_PATH,
    MODELS_DIR,
    DATASOURCE_NAME,
)

build_shop_duckdb(DB_PATH)
GOLD = compute_gold(DB_PATH)
print("reference answers:", {k: GOLD[k] for k in ("total", "aov", "cust_reach", "rev_plus_pop")})

reference answers: {'total': 1400.0, 'aov': 233.33333333333334, 'cust_reach': 466.6666666666667, 'rev_plus_pop': 4400.0}


## Step 2 — Ingest the OSI config with the CLI

Two commands — exactly what you'd type in a shell (or an agent would run for you):

1. **Register the datasource.** `import-osi` reads real column *types* by introspecting the database live, so the datasource has to exist first — this is the separate command that must run before the import.
2. **Import the OSI config** into that datasource.

We point both at this demo's storage folder with the `SLAYER_STORAGE` environment variable — the same folder the MCP server reads in Step 3.

In [2]:
env = {**os.environ, "SLAYER_STORAGE": str(MODELS_DIR)}
shutil.rmtree(MODELS_DIR, ignore_errors=True)  # start from a clean store


def slayer(*args):
    """Run a `slayer` CLI command against the demo storage; echo it like a shell."""
    print("$ slayer", *args)
    result = subprocess.run(["slayer", *map(str, args)], capture_output=True, text=True, env=env)
    print(result.stdout, end="")
    if result.returncode != 0:
        print(result.stderr)
        raise RuntimeError(f"slayer exited {result.returncode}")
    print()


slayer("datasources", "create", f"duckdb:///{os.path.relpath(DB_PATH)}", "--name", DATASOURCE_NAME, "-y")
slayer("import-osi", os.path.relpath(OSI_CONFIG), "--datasource", DATASOURCE_NAME)

$ slayer datasources create duckdb:///.cache/shop.duckdb --name shop_osi -y


Created datasource 'shop_osi' (duckdb).

$ slayer import-osi shop.osi.yaml --datasource shop_osi


Imported model: orders (8 columns, 7 measures)
Imported model: customers (4 columns, 0 measures)
Imported model: products (3 columns, 0 measures)
Imported model: regions (3 columns, 0 measures)

Done: 4 models, 0 unconverted, 0 dropped



## Step 3 — Connect the MCP server

The models exist now. We spin up SLayer's MCP server **in-process** against the same storage the CLI just wrote, and list its tools — the same set Claude sees over stdio.

In [3]:
from slayer.mcp.server import create_mcp_server
from slayer.storage.yaml_storage import YAMLStorage

mcp = create_mcp_server(storage=YAMLStorage(base_dir=str(MODELS_DIR)))
tools = await mcp.list_tools()
print(f"SLayer MCP exposes {len(tools)} tools:")
print(", ".join(t.name for t in tools))


async def call(name, **arguments):
    """Invoke an MCP tool and return its text response (what an agent receives)."""
    content, _ = await mcp.call_tool(name=name, arguments=arguments)
    return content[0].text

SLayer MCP exposes 21 tools:
query, query_nested, models_summary, inspect_model, inspect, create_model, edit_model, create_datasource, list_datasources, describe_datasource, edit_datasource, delete_model, validate_models, recommend_root_model, delete_datasource, ingest_datasource_models, set_datasource_priority, get_datasource_priority, save_memory, forget_memory, search


## Step 4 — Explore like an agent

Before querying, an agent orients itself: `models_summary` for the lay of the land, then `inspect` to drill into a model — and a single column, where the OSI `ai_context` shows up as descriptions and synonyms.

In [4]:
print(await call("models_summary", datasource_name=DATASOURCE_NAME))

# Datasource: `shop_osi` — 4 model(s)

## `customers`
Columns: 4
Measures: 
Joins to: `regions`

## `orders`
Order line items
One row per order.
Synonyms: sales, purchases
Columns: 7
Measures: total_amount, order_count, aov, revenue_line, cust_reach, rev_plus_pop, bridge_metric
Joins to: `customers`, `products`

## `products`
Columns: 3
Measures: 
Joins to: _(none)_

## `regions`
Columns: 3
Measures: 
Joins to: _(none)_


In [5]:
print(await call("inspect", reference="orders", entity_type="model"))

# `orders`
Order line items
One row per order.
Synonyms: sales, purchases
Columns: order_id, customer_id, product_id, amount, quantity, ordered_at, status
Measures: total_amount, order_count, aov, revenue_line, cust_reach, rev_plus_pop, bridge_metric
Aggregations: _(none)_
Joins to: customers, products


The OSI `ai_context` and `custom_extensions` rode along into the imported model. Drill into the `amount` column to see them — description, synonyms, label, and a live sample range:

In [6]:
print(await call("inspect", reference="orders.amount", entity_type="column", compact=False))

Column: shop_osi.orders.amount
Type: DOUBLE
Description: Gross order value in USD.
Synonyms: revenue, gross
Label: Order amount
Format: type=<NumberFormatType.FLOAT: 'float'> precision=None symbol=None
SQL: amount
Sample values: 100.0 .. 400.0


### Semantic discovery with `search`

An agent that doesn't know the measure name can `search` for it. Scoping to the datasource keeps hits on this shop. (The embedding channel is off here — no API key — so this runs on BM25 + full-text alone, which still finds it.)

In [7]:
hits = json.loads(await call("search", question="average order value", datasource=DATASOURCE_NAME, max_results=5))
for h in hits["results"]:
    print(f"  {h['kind']:<8} {h.get('id') or h.get('canonical_id')}   (score {h['score']:.3f})")

found = [h.get("id") or h.get("canonical_id") for h in hits["results"]]
assert any(str(x).endswith("aov") for x in found), found
print("\nOK — search surfaced the `aov` measure")

  measure  shop_osi.orders.aov   (score 0.016)
  column   shop_osi.orders.amount   (score 0.016)
  measure  shop_osi.orders.total_amount   (score 0.016)
  column   shop_osi.orders.ordered_at   (score 0.016)
  measure  shop_osi.orders.order_count   (score 0.015)

OK — search surfaced the `aov` measure


## Step 5 — Query via the MCP `query` tool

Now the agent issues `query` calls — the same JSON-RPC an LLM emits — and we check each answer against the gold values from Step 1. `format="json"` returns the rows followed by a human-readable attributes block; `raw_decode` reads just the leading JSON.

In [8]:
async def query_rows(**arguments):
    text = await call("query", format="json", **arguments)
    rows, _ = json.JSONDecoder().raw_decode(text.strip())
    return rows


# Q1 — a simple metric
rows = await query_rows(source_model="orders", measures=[{"formula": "total_amount"}])
print("total_amount:", rows, " gold:", GOLD["total"])
assert abs(rows[0]["orders.total_amount"] - GOLD["total"]) < 1e-9

# Q2 — grouped by a multi-hop joined dimension
rows = await query_rows(
    source_model="orders",
    measures=[{"formula": "total_amount"}],
    dimensions=["customers.regions.name"],
    order=[{"column": "customers.regions.name"}],
)
display(pd.DataFrame(rows))
pairs = [(r["orders.customers.regions.name"], r["orders.total_amount"]) for r in rows]
assert pairs == [(g["region"], g["amount"]) for g in GOLD["by_region"]]

# Q3 — a cross-dataset metric
rows = await query_rows(source_model="orders", measures=[{"formula": "cust_reach"}])
print("cust_reach:  ", rows, " gold:", GOLD["cust_reach"])
assert abs(rows[0]["orders.cust_reach"] - GOLD["cust_reach"]) < 1e-9

print("\nOK — every MCP query matches gold")

total_amount: [{'orders.total_amount': 1400.0}]  gold: 1400.0


,orders.customers.regions.name,orders.total_amount
0,North,750.0
1,South,650.0


cust_reach:   [{'orders.cust_reach': 466.6666666666667}]  gold: 466.6666666666667

OK — every MCP query matches gold


## Recap

The same OSI import as the [library notebook](osi_import_nb.ipynb), driven the way an agent lives it:

- **two CLI commands** — `slayer datasources create` (the datasource the importer introspects) then `slayer import-osi`,
- **MCP tools** — `models_summary` / `inspect` / `search` to explore (OSI `ai_context` surfacing as descriptions + synonyms), then `query` to answer questions,
- every answer checked against the shared gold SQL in `setup_osi.compute_gold`.

### Further reading

- [MCP Setup — AI Agents](../../getting-started/mcp.md) — registering SLayer with Claude Code and other agents.
- [Introspecting a datasource via MCP](../08_mcp_introspect/mcp_introspect_nb.ipynb) — a deeper tour of the introspection tools.
- [Importing OSI configs](../../osi/osi_import.md) — the full conversion reference.